## Phase 3: The two models

### Load

In [3]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.metrics import evaluate, build_test_liked

LIKE_THRESHOLD = 4.0
K = 10

train = pd.read_parquet("../data/train.parquet")
test  = pd.read_parquet("../data/test.parquet")
test_liked = build_test_liked(test, LIKE_THRESHOLD)

print(f"train {len(train):,} test {len(test):,} evaluable users {len(test_liked):,}")

train 22,261,221 test 414,622 evaluable users 6,269


### Build the user-movie matrices

In [5]:
from scipy.sparse import csr_matrix

# Give every user a row number and every movie a column number
user_ids = np.sort(train.userId.unique())
item_ids = np.sort(train.movieId.unique())
u_index = pd.Series(np.arange(len(user_ids)), index=user_ids)
i_index = pd.Series(np.arange(len(item_ids)), index=item_ids)

def to_matrix(df):
    """Rows = users, columns = movies, 1 where the pair appears in df."""
    return csr_matrix(
        (np.ones(len(df), dtype="float32"),
         (u_index[df.userId].values, i_index[df.movieId].values)),
        shape=(len(user_ids), len(item_ids)))

R_seen  = to_matrix(train)                                   # everything they rated
R_liked = to_matrix(train[train.rating >= LIKE_THRESHOLD])   # only what they liked

print(f"users {R_liked.shape[0]:,}  movies {R_liked.shape[1]:,}  "
      f"likes {R_liked.nnz:,}  density {R_liked.nnz / np.prod(R_liked.shape):.4%}")

users 150,543  movies 12,837  likes 11,142,564  density 0.5766%


In [6]:
R_liked.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]],
      shape=(150543, 12837), dtype=float32)

### Pick the top 10, skipping movies they've already rated

In [8]:
def top_k_unseen(scores, user_id, k):
    scores = scores.astype("float64").copy()
    scores[R_seen[u_index[user_id]].indices] = -np.inf   # never recommend what they've seen
    top = np.argsort(-scores)[:k]                        # highest scores first
    return item_ids[top].tolist()

### Popularity

In [10]:
# Count how many users liked each movie
pop_scores = np.asarray(R_liked.sum(axis=0)).ravel()

def popularity_rec(user_id, k):
    return top_k_unseen(pop_scores, user_id, k)

print(evaluate(popularity_rec, test_liked, k=K, n_users=1000).to_string())

recall       0.0307
precision    0.0650
ndcg         0.0789


### Item-item

In [12]:
from sklearn.preprocessing import normalize

R_norm = normalize(R_liked, axis=0)   # so blockbusters don't win every similarity

def itemitem_rec(user_id, k):
    """Score each movie by how much it overlaps with what this user liked."""
    liked = np.asarray(R_liked[u_index[user_id]].todense()).ravel()
    scores = R_norm.T @ (R_norm @ liked)
    return top_k_unseen(scores, user_id, k)

print(evaluate(itemitem_rec, test_liked, k=K, n_users=1000).to_string())

recall       0.0429
precision    0.0980
ndcg         0.1105


### Compare

In [14]:
results = pd.DataFrame({
    "Popularity": evaluate(popularity_rec, test_liked, k=K, n_users=1000),
    "Item-item":  evaluate(itemitem_rec, test_liked, k=K, n_users=1000),
}).T

results

,recall,precision,ndcg
Popularity,0.0307,0.065,0.0789
Item-item,0.0429,0.098,0.1105


In [15]:
# Same comparison, different 1,000 users. The gap should hold if it's real.
check = pd.DataFrame({
    "Popularity": evaluate(popularity_rec, test_liked, k=K, n_users=1000, seed=7),
    "Item-item":  evaluate(itemitem_rec, test_liked, k=K, n_users=1000, seed=7),
}).T

check

,recall,precision,ndcg
Popularity,0.0323,0.0631,0.0769
Item-item,0.0485,0.0963,0.1145
